# Portfolio Optimization Notebook
This notebook separates function definitions and execution logic so you can use `%time` to time only the optimization step.

In [ ]:
# === Cell 1: Imports and Global Config ===
import os
import pandas as pd
import numpy as np
from numba import njit
from scipy.optimize import minimize
from scipy.optimize import Bounds, LinearConstraint
from joblib import Parallel, delayed
import multiprocessing
import time
import matplotlib.pyplot as plt

In [ ]:
# === Cell 2: Monte Carlo Functions ===
def simulate_single_run(mean_returns, cov_matrix, risk_free_rate, num_assets):
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)
    ret = np.dot(weights, mean_returns)
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe = (ret - risk_free_rate) / vol
    return weights, ret, vol, sharpe

def monte_carlo_portfolio_optimization_opt(df, n_simulations=100_000, n_assets_to_select=50, risk_free_rate=0.02, random_seed=42, fixed_tickers=None):
    s_t = time.time()
    np.random.seed(random_seed)
    if fixed_tickers is not None:
        selected_tickers = fixed_tickers
    else:
        selected_tickers = np.random.choice(df.columns.tolist(), size=n_assets_to_select, replace=False)
    df_selected = df[selected_tickers]
    mean_returns = df_selected.mean().values * 252
    cov_matrix = df_selected.cov().values * 252
    num_assets = len(selected_tickers)
    results = Parallel(n_jobs=multiprocessing.cpu_count(), backend="loky")(
        delayed(simulate_single_run)(mean_returns, cov_matrix, risk_free_rate, num_assets)
        for _ in range(n_simulations)
    )
    all_weights, ret_arr, vol_arr, sharpe_arr = zip(*results)
    max_idx = np.argmax(sharpe_arr)
    print("MC Runtime:", time.time() - s_t)
    return {
        "tickers": selected_tickers,
        "max_sharpe": sharpe_arr[max_idx],
        "expected_return": ret_arr[max_idx],
        "expected_volatility": vol_arr[max_idx],
        "optimal_weights": all_weights[max_idx]
    }

In [ ]:
# === Cell 3: MVO Optimization ===
def portfolio_metrics(weights, mean_returns, cov_matrix, risk_free_rate):
    port_return = np.dot(weights, mean_returns)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (port_return - risk_free_rate) / port_vol
    return -sharpe_ratio

def weight_constraints(n):
    bounds = Bounds(0, 1)
    linear_constraint = LinearConstraint(np.ones(n), lb=1, ub=1)
    return bounds, linear_constraint

def optimize_portfolio_from_weights(df, init_weights, tickers, risk_free_rate=0.02):
    df = df[tickers]
    mean_returns = df.mean() * 252
    cov_matrix = df.cov() * 252
    bounds, constraint = weight_constraints(len(tickers))
    result = minimize(
        fun=portfolio_metrics,
        x0=np.array(init_weights),
        args=(mean_returns, cov_matrix, risk_free_rate),
        method='SLSQP',
        bounds=bounds,
        constraints=[constraint],
        options={'disp': False}
    )
    return {
        "optimized_weights": result.x,
        "sharpe": -result.fun,
        "expected_return": np.dot(result.x, mean_returns),
        "expected_volatility": np.sqrt(np.dot(result.x.T, np.dot(cov_matrix, result.x))),
        "tickers": tickers
    }

In [ ]:
# === Cell 4: Main Optimization Runner ===
def run_portfolio_optimization(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    risk_free = df['SHY'].mean()
    df = df.drop(columns=['SHY'])

    best_results = []
    for seed in range(50):
        result = monte_carlo_portfolio_optimization_opt(
            df=df,
            n_simulations=5000,
            n_assets_to_select=50,
            risk_free_rate=risk_free,
            random_seed=seed
        )
        best_results.append(result)
        best_results = sorted(best_results, key=lambda x: x['max_sharpe'], reverse=True)[:10]

    mvo_results = []
    for r in best_results:
        opt = optimize_portfolio_from_weights(
            df=df,
            init_weights=r['optimal_weights'],
            tickers=r['tickers'],
            risk_free_rate=risk_free
        )
        mvo_results.append(opt)

    return max(mvo_results, key=lambda x: x['sharpe'])

In [ ]:
# === Cell 5: Run with Timing ===
# %time result = run_portfolio_optimization("./data/raw/processed.csv")
# print(result)